# OpenPlaque — User-Friendly Plaque + PCAT Visualization v2

This notebook improves the presentation layer without changing the validated quantitative endpoints.

**Changes from v1**
- Plaque hotspot ranking is anatomically constrained to plaque near the nnU-Net vessel label, so remote chest-wall/bone false-positive islands are not promoted as hotspots. This filter is **for visualization only** and does not change canonical TPV.
- RCA PCAT is shown in **artery-centered planes perpendicular to the frozen centerline**, with lumen, modeled outer wall, and canonical PCAT shell drawn explicitly.
- The dashboard labels the circular-margin sensitivity separately and, when available, includes the directional-interface result in the full tested geometry range.
- A compact top-plaque-regions table is exported.

PCAT attenuation is an imaging surrogate related to perivascular inflammation; it is not a direct inflammation measurement and this is not Caristo FAI-Score. Research use only.


In [ ]:
# FIRST EXECUTABLE CELL — mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch user-friendly-visualization-v2-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.ndimage import map_coordinates
from IPython.display import display
REPO=Path('/content/OpenPlaque'); sys.path.insert(0,str(REPO/'src'))
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'User_Friendly_Plaque_PCAT_Report_v2'; OUT.mkdir(parents=True,exist_ok=True)
os.environ['nnUNet_raw']='/content/nnUNet_raw'; os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'; os.environ['nnUNet_results']='/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'],os.environ['nnUNet_preprocessed'],os.environ['nnUNet_results']]: Path(d).mkdir(parents=True,exist_ok=True)
model_zip=ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'; model_target=Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists(): raise FileNotFoundError(model_zip)
    with zipfile.ZipFile(model_zip) as z: z.extractall('/content/nnUNet_results')
drive_zip=ROOT/'Full_DICOM.zip'; local_zip=Path('/content/Full_DICOM.zip')
if not drive_zip.exists(): raise FileNotFoundError(drive_zip)
if not local_zip.exists() or local_zip.stat().st_size!=drive_zip.stat().st_size: shutil.copyfile(drive_zip,local_zip)
from openplaque.study import OpenPlaqueStudy
shutil.rmtree('/content/full_dicom_friendly_v2',ignore_errors=True)
study=OpenPlaqueStudy(str(local_zip),extract_root='/content/full_dicom_friendly_v2')
print('Output:',OUT)


## 1. Recreate canonical plaque segmentations and make an anatomy-constrained display mask

Canonical TPV is unchanged. For display and hotspot selection only, plaque voxels must lie within a physical-distance neighborhood of the predicted vessel label.


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series
fallback={'RCA':1035,'LCX':1039,'LAD':1043}
series_map,_=detect_artery_series(study,fallback_series=fallback,return_candidates=True)
reports=[]
for vessel in ['LAD','RCA','LCX']:
    image,volume,_=study.load_series(series_map[vessel])
    print('Segmenting',vessel,'series',series_map[vessel])
    reports.append(segment_vessel(image,volume,vessel))
def canonical_refine(r):
    return refine_plaque_mask(volume=r.volume,mask=r.mask,spacing=r.mask_image.GetSpacing(),remove_small=True,min_component_voxels=10,trim_lumen_adjacent=True,lumen_distance_voxels=1,erode_core=False,high_hu_threshold=None,low_hu_threshold=None)
canonical={r.name:canonical_refine(r) for r in reports}
DISPLAY_MAX_DISTANCE_MM=3.0
display_masks={}; stats=[]
for r in reports:
    vessel=(r.mask==1); plaque=(canonical[r.name].refined_mask==2); sp_zyx=np.array(r.mask_image.GetSpacing(),float)[::-1]
    dist=ndi.distance_transform_edt(~vessel,sampling=sp_zyx); keep=plaque&(dist<=DISPLAY_MAX_DISTANCE_MM); display_masks[r.name]=keep
    stats.append({'vessel':r.name,'canonical_plaque_voxels':int(plaque.sum()),'display_near_vessel_voxels':int(keep.sum()),'display_retained_pct':100*keep.sum()/max(1,plaque.sum())})
display_stats=pd.DataFrame(stats); display_stats.to_csv(OUT/'display_filter_qc.csv',index=False); display(display_stats)


In [ ]:
hotspot_rows=[]; fig,axs=plt.subplots(3,3,figsize=(14,14))
for row,r in enumerate(reports):
    m=display_masks[r.name]; counts=np.sum(m,axis=(1,2)); order=np.argsort(counts)[::-1]; chosen=[]
    for z in order:
        if counts[z]<=0: break
        if all(abs(int(z)-q)>=3 for q in chosen): chosen.append(int(z))
        if len(chosen)==3: break
    while len(chosen)<3: chosen.append(r.volume.shape[0]//2)
    voxel_mm3=float(np.prod(r.mask_image.GetSpacing()))
    for col,z in enumerate(chosen):
        ax=axs[row,col]; ax.imshow(r.volume[z],cmap='gray',vmin=-200,vmax=800); overlay=np.ma.masked_where(~m[z],m[z]); ax.imshow(overlay,alpha=.55,cmap='autumn',interpolation='nearest')
        if np.any(m[z]): ax.contour(m[z],levels=[0.5],linewidths=1.2)
        ax.set_title(f'{r.name} plaque region {col+1}\nframe {z}, local plaque {counts[z]*voxel_mm3:.1f} mm³'); ax.axis('off')
        hotspot_rows.append({'vessel':r.name,'rank':col+1,'frame':z,'local_plaque_voxels':int(counts[z]),'local_plaque_mm3':float(counts[z]*voxel_mm3)})
fig.suptitle('Top coronary-adjacent plaque regions (visualization filter only)',fontsize=16); plt.tight_layout(rect=[0,0,1,.97]); plt.savefig(OUT/'01_plaque_hotspot_gallery_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
hotspots=pd.DataFrame(hotspot_rows); hotspots.to_csv(OUT/'top_plaque_regions.csv',index=False); display(hotspots)


## 2. Artery-centered RCA PCAT cross-sections


In [ ]:
BASE=ROOT/'PCAT_RCA_10_50'; cp=BASE/'rca_centerline_smoothed_zyx.csv'; rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists(): raise FileNotFoundError('Missing frozen RCA PCAT centerline/radius inputs.')
source_img,ct,_=study.load_series(7); ct=np.asarray(ct,float); sp_xyz=np.array(source_img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]
cl=pd.read_csv(cp); rad=pd.read_csv(rp); arc=cl.arc_mm.to_numpy(float); pts_zyx=cl[['z','y','x']].to_numpy(float); pts_mm=pts_zyx*sp_zyx; lumen=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))
aorta_candidates=[ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz']; ap=next((p for p in aorta_candidates if p.exists()),None)
if ap is None: raise FileNotFoundError('Missing TotalSegmentator aorta mask.')
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()): ai=sitk.Resample(ai,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai).astype(float)
def plane_basis(t):
    t=np.asarray(t,float); t=t/np.linalg.norm(t); ref=np.array([1.,0.,0.]) if abs(t[0])<.85 else np.array([0.,1.,0.]); u=np.cross(t,ref); u=u/np.linalg.norm(u); v=np.cross(t,u); return u,v/np.linalg.norm(v)
def sample_plane(target,half=8.,pix=.20):
    i=int(np.argmin(np.abs(arc-target))); i0=max(0,i-2); i1=min(len(arc)-1,i+2); tangent=pts_mm[i1]-pts_mm[i0]; tangent=tangent/np.linalg.norm(tangent); u,v=plane_basis(tangent); center=pts_mm[i]
    c=np.arange(-half,half+1e-9,pix); U,V=np.meshgrid(c,c,indexing='xy'); xyz=center[None,None,:]+U[...,None]*u+V[...,None]*v; vox=(xyz/sp_zyx).reshape(-1,3).T
    img=map_coordinates(ct,vox,order=1,mode='nearest').reshape(U.shape); am=map_coordinates(aorta,vox,order=0,mode='nearest').reshape(U.shape)>0.5; rr=np.sqrt(U**2+V**2); lum=float(lumen[i]); outer=lum+.75; shell_outer=3*outer; fat=(rr>outer)&(rr<=shell_outer)&(~am)&(img>=-190)&(img<=-30)
    return {'arc':float(arc[i]),'img':img,'fat':fat,'lumen':lum,'outer':outer,'shell_outer':shell_outer,'extent':[-half,half,-half,half]}
planes=[sample_plane(x) for x in [10,20,30,40,50]]
fig,axs=plt.subplots(1,5,figsize=(20,4.5))
for ax,d,target in zip(axs,planes,[10,20,30,40,50]):
    ax.imshow(d['img'],cmap='gray',vmin=-200,vmax=800,extent=d['extent'],origin='lower'); fi=np.ma.masked_where(~d['fat'],d['img']); im=ax.imshow(fi,cmap='coolwarm',vmin=-120,vmax=-60,alpha=.8,extent=d['extent'],origin='lower')
    for rr,ls in [(d['lumen'],'-'),(d['outer'],'--'),(d['shell_outer'],':')]: ax.add_patch(plt.Circle((0,0),rr,fill=False,linestyle=ls,linewidth=1.7))
    ax.plot(0,0,'+',markersize=9); ax.set_title(f'RCA {target} mm\nactual {d["arc"]:.1f} mm'); ax.set_xlim(-8,8); ax.set_ylim(-8,8); ax.set_aspect('equal'); ax.set_xlabel('mm')
axs[0].set_ylabel('mm'); fig.suptitle('RCA artery-centered PCAT: solid=lumen, dashed=modeled outer wall, dotted=shell edge',fontsize=13); cb=fig.colorbar(im,ax=axs.ravel().tolist(),shrink=.78,pad=.02); cb.set_label('PCAT attenuation (HU)'); plt.savefig(OUT/'02_rca_pcat_cross_sections_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 3. Longitudinal ribbon and corrected dashboard


In [ ]:
lp=ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_canonical_longitudinal_v2.csv'
if not lp.exists(): lp=ROOT/'PCAT_RCA_10_50_Reproducibility_Lock'/'pcat_canonical_primary_longitudinal.csv'
longdf=pd.read_csv(lp); x=(longdf.arc_start_mm.to_numpy(float)+longdf.arc_end_mm.to_numpy(float))/2; y=longdf.mean_hu.to_numpy(float)
fig,ax=plt.subplots(figsize=(12,3.2)); sc=ax.scatter(x,np.zeros_like(x),c=y,cmap='coolwarm',vmin=-110,vmax=-75,s=260,marker='s'); ax.set_xlim(10,50); ax.set_yticks([]); ax.set_xlabel('Distance from RCA ostium (mm)'); ax.set_title('RCA longitudinal PCAT attenuation ribbon'); cb=fig.colorbar(sc,ax=ax,pad=.02); cb.set_label('Mean HU per 1-mm segment'); plt.tight_layout(); plt.savefig(OUT/'03_rca_longitudinal_pcat_ribbon_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
tpv=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'tpv_metrics_by_vessel_v2.csv'); pcat=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_canonical_primary_v2.csv'); ps=pd.read_csv(ROOT/'Combined_TPV_PCAT_All_Metrics_v2'/'pcat_circular_sensitivity_v2.csv')
total=tpv[tpv.vessel=='TOTAL'].iloc[0]; primary=pcat.iloc[0]; cmin=float(ps.pcat_mean_hu.min()); cmax=float(ps.pcat_mean_hu.max()); directional=np.nan; dp=ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv'
if dp.exists():
    d=pd.read_csv(dp); directional=float(d.iloc[0]['directional_pcat_mean_hu']) if 'directional_pcat_mean_hu' in d.columns else np.nan
fmin=min(cmin,directional) if np.isfinite(directional) else cmin; fmax=max(cmax,directional) if np.isfinite(directional) else cmax
fig=plt.figure(figsize=(13,8)); gs=fig.add_gridspec(2,2,height_ratios=[1,1.1]); ax1=fig.add_subplot(gs[0,0]); vv=tpv[tpv.vessel!='TOTAL']; ax1.bar(vv.vessel,vv.canonical_refined_tpv_mm3); ax1.set_ylabel('Refined TPV (mm³)'); ax1.set_title('Plaque volume by artery')
ax2=fig.add_subplot(gs[0,1]); ax2.axis('off'); txt=f'Total refined TPV: {total.canonical_refined_tpv_mm3:.0f} mm³\nRaw TPV: {total.raw_tpv_mm3:.0f} mm³\nTPV sensitivity: {total.sensitivity_min_mm3:.0f}–{total.sensitivity_max_mm3:.0f} mm³\n\nRCA 10–50 mm PCAT: {primary.pcat_mean_hu:.2f} HU\nCircular-margin range: {cmin:.2f} to {cmax:.2f} HU\nFull tested geometry range: {fmin:.2f} to {fmax:.2f} HU'; ax2.text(.03,.95,txt,va='top',fontsize=13,bbox=dict(boxstyle='round',alpha=.08))
ax3=fig.add_subplot(gs[1,:]); ax3.plot(x,y,marker='o',markersize=3); ax3.axhline(float(primary.pcat_mean_hu),linestyle='--',label='RCA 10–50 mean'); ax3.set_xlabel('Distance from RCA ostium (mm)'); ax3.set_ylabel('PCAT HU'); ax3.set_title('Longitudinal RCA PCAT attenuation'); ax3.legend(); fig.suptitle('OpenPlaque plaque + PCAT summary',fontsize=18); plt.tight_layout(rect=[0,0,1,.96]); plt.savefig(OUT/'04_summary_dashboard_v2.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


In [ ]:
summary=pd.DataFrame([{'total_refined_tpv_mm3':float(total.canonical_refined_tpv_mm3),'raw_tpv_mm3':float(total.raw_tpv_mm3),'tpv_sensitivity_min_mm3':float(total.sensitivity_min_mm3),'tpv_sensitivity_max_mm3':float(total.sensitivity_max_mm3),'rca_pcat_mean_hu':float(primary.pcat_mean_hu),'circular_geometry_min_hu':cmin,'circular_geometry_max_hu':cmax,'full_tested_geometry_min_hu':fmin,'full_tested_geometry_max_hu':fmax,'directional_interface_mean_hu':directional}]); summary.to_csv(OUT/'friendly_report_summary_v2.csv',index=False)
import base64
def img64(path): return base64.b64encode(Path(path).read_bytes()).decode()
imgs=['01_plaque_hotspot_gallery_v2.png','02_rca_pcat_cross_sections_v2.png','03_rca_longitudinal_pcat_ribbon_v2.png','04_summary_dashboard_v2.png']; html=['<html><head><meta charset="utf-8"><title>OpenPlaque Friendly Visual Report v2</title></head><body style="font-family:Arial;max-width:1200px;margin:auto">','<h1>OpenPlaque Plaque + PCAT Visual Report v2</h1>','<p><b>Research use only.</b> PCAT attenuation is an imaging surrogate associated with perivascular inflammation, not a direct inflammation measurement.</p>']
for name in imgs: html.append(f'<h2>{name}</h2><img style="max-width:100%;height:auto" src="data:image/png;base64,{img64(OUT/name)}">')
html.append('<h2>Top plaque regions</h2>'+hotspots.to_html(index=False,float_format=lambda z:f'{z:.1f}')); html.append('<h2>Display-filter QC</h2>'+display_stats.to_html(index=False,float_format=lambda z:f'{z:.1f}')); html.append('<h2>Summary metrics</h2>'+summary.to_html(index=False,float_format=lambda z:f'{z:.2f}')); html.append('</body></html>'); (OUT/'OPENPLAQUE_FRIENDLY_VISUAL_REPORT_V2.html').write_text(''.join(html),encoding='utf-8')
zip_path=OUT/'OPENPLAQUE_FRIENDLY_VISUAL_V2_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=zip_path: z.write(p,arcname=p.name)
print('Report-back ZIP:',zip_path)
